<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - Study 2 - Phase 4 - Elaine

This phase aims at compiling the target corpus for Lexical Multi-Dimensional Analysis.

## Import the Target Corpus into a DataFrame

In [1]:
import pandas as pd

df_ted_tc = pd.read_json("../cl_st2_ph3_elaine/cl_st2_ph3_elaine/cl_st2_ph3_elaine_tc.jsonl", lines=True)

## Identify nonverbal event markers

In [2]:
# Task: From df_ted_tc, extract all substrings enclosed in parentheses in the Text column, and create a set of unique phrases (including the parentheses).

# Ensure df_ted_tc exists with a 'Text' column

import re

if 'df_ted_tc' in globals() and isinstance(df_ted_tc, pd.DataFrame) and 'Text' in df_ted_tc.columns:
    # Regex to capture non-greedy content between parentheses, handling multiple per row
    pattern = re.compile(r'\([^()]*\)')
    # Extract lists of matches per row (skip NaN safely), then flatten
    all_paren_phrases = (
        df_ted_tc['Text']
        .dropna()
        .astype(str)
        .apply(pattern.findall)
        .explode()
        .dropna()
        .tolist()
    )
    phrases_in_parentheses = set(all_paren_phrases)

    print(f"Unique parenthetical phrases found: {len(phrases_in_parentheses)}")
else:
    phrases_in_parentheses = set()
    print("df_ted_tc with column 'Text' is not available.")

# Optionally preview a few items deterministically
#preview = sorted(list(phrases_in_parentheses))[:20]
#preview

Unique parenthetical phrases found: 1321


In [3]:
# Task: From the existing `phrases_in_parentheses` set, build a DataFrame with counts and export it as JSON.

import pandas as pd

# Validate source set
if 'phrases_in_parentheses' in globals() and isinstance(phrases_in_parentheses, set):
    # We also need counts per phrase from the original list of occurrences
    if 'all_paren_phrases' in globals() and isinstance(all_paren_phrases, list):
        # Build counts using pandas
        s = pd.Series(all_paren_phrases, dtype="string").dropna()
        df_phrases_in_parentheses = (
            s.value_counts()
            .rename_axis('Phrases in Parentheses')
            .reset_index(name='Count')
        )
    else:
        # Fall back: create counts = 1 for each unique phrase if the full list isn't available
        df_phrases_in_parentheses = pd.DataFrame(
            {'Phrases in Parentheses': sorted(list(phrases_in_parentheses)),
             'Count': 1}
        )
else:
    # Empty fallback
    df_phrases_in_parentheses = pd.DataFrame(
        {'Phrases in Parentheses': pd.Series(dtype="string"),
         'Count': pd.Series(dtype="int")}
    )

# Preview
#df_phrases_in_parentheses

# Export to JSON (records)
export_json_path = "corpus/00_sources/phrases_in_parentheses_counts.json"
df_phrases_in_parentheses.to_json(export_json_path, orient='records', force_ascii=False)
print(f"Exported df_phrases_in_parentheses to {export_json_path} with {len(df_phrases_in_parentheses)} rows.")

Exported df_phrases_in_parentheses to corpus/00_sources/phrases_in_parentheses_counts.json with 1321 rows.


### Inspect one row

In [4]:
print(df_ted_tc.at[3319, 'Text'])

The ways of the world often baffle me. I sometimes wonder if I missed the memo about the most basic things. What are you supposed to make for dinner? What do you talk about in an elevator? Why do people cut in line? How do you leave a dinner party without being rude? Or do you leave it all? ["Why are you still here?"] (Laughter) My tendency to see the world like I'm from outer space was a bit of a liability when I was a kid. True story. ["What's a noogie?"] (Laughter) But it's been helpful in my career. I'm a cartoonist. When I first started making cartoons for The New Yorker about a decade ago, I kept my ideas light and quirky. And please shout out if you don't get any of them, I'll explain. (Laughter) ["The synchronized swim team quits"] I didn't draw anything too personal. ["Unicorns do exist!"] I mean, personal for someone. I figured I was too specific, too hard to relate to and read, possibly, too female. It took a breakup. (Laughter) ["I know there would be a time I could wear th

After examining the frequency of nonlexical event annotations, we decided to exclude them from the analysis as they may distort the Lexical Multi-Dimensional Analysis.

```
  {
    "Phrases in Parentheses": "(Laughter)",
    "Count": 15108
  },
  {
    "Phrases in Parentheses": "(Applause)",
    "Count": 8059
  },
```

In [5]:
# Task: Remove all occurrences of phrases from df_phrases_in_parentheses['Phrases in Parentheses']
# from df_ted_tc['Text'].

# Preconditions check
required_dfs_ok = (
        'df_ted_tc' in globals() and isinstance(df_ted_tc, pd.DataFrame) and 'Text' in df_ted_tc.columns and
        'df_phrases_in_parentheses' in globals() and isinstance(df_phrases_in_parentheses, pd.DataFrame) and
        'Phrases in Parentheses' in df_phrases_in_parentheses.columns
)

if required_dfs_ok:
    # Build an efficient single regex pattern that matches any of the phrases literally
    phrases = (
        df_phrases_in_parentheses['Phrases in Parentheses']
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )
    if phrases:
        # Escape each phrase to avoid regex meta-character issues
        escaped = [re.escape(p) for p in phrases]
        # Join with alternation; use non-capturing group for safety
        pattern_parentheses_phrases = re.compile(r'(?:' + '|'.join(escaped) + r')')

        # Replace all occurrences with empty string; preserve NaNs
        df_ted_tc['Text'] = df_ted_tc['Text'].astype(str).str.replace(pattern_parentheses_phrases, '', regex=True)

        # Optional: collapse extra spaces introduced by removals and trim
        df_ted_tc['Text'] = df_ted_tc['Text'].str.replace(r'\s+', ' ', regex=True).str.strip()
    else:
        print("No phrases provided to remove.")
else:
    print(
        "Required DataFrames or columns are missing: df_ted_tc['Text'] and df_phrases_in_parentheses['Phrases in Parentheses'].")

### Reinspect the row

In [6]:
print(df_ted_tc.at[3319, 'Text'])

The ways of the world often baffle me. I sometimes wonder if I missed the memo about the most basic things. What are you supposed to make for dinner? What do you talk about in an elevator? Why do people cut in line? How do you leave a dinner party without being rude? Or do you leave it all? ["Why are you still here?"] My tendency to see the world like I'm from outer space was a bit of a liability when I was a kid. True story. ["What's a noogie?"] But it's been helpful in my career. I'm a cartoonist. When I first started making cartoons for The New Yorker about a decade ago, I kept my ideas light and quirky. And please shout out if you don't get any of them, I'll explain. ["The synchronized swim team quits"] I didn't draw anything too personal. ["Unicorns do exist!"] I mean, personal for someone. I figured I was too specific, too hard to relate to and read, possibly, too female. It took a breakup. ["I know there would be a time I could wear them without destroying my feet."] It took a b

## Tokenise

Please refer to [What is tokenization in NLP?](https://www.analyticsvidhya.com/blog/2020/05/what-is-tokenization-nlp/).

### Inspect one row

In [7]:
print(df_ted_tc.at[3319, 'Text'])

The ways of the world often baffle me. I sometimes wonder if I missed the memo about the most basic things. What are you supposed to make for dinner? What do you talk about in an elevator? Why do people cut in line? How do you leave a dinner party without being rude? Or do you leave it all? ["Why are you still here?"] My tendency to see the world like I'm from outer space was a bit of a liability when I was a kid. True story. ["What's a noogie?"] But it's been helpful in my career. I'm a cartoonist. When I first started making cartoons for The New Yorker about a decade ago, I kept my ideas light and quirky. And please shout out if you don't get any of them, I'll explain. ["The synchronized swim team quits"] I didn't draw anything too personal. ["Unicorns do exist!"] I mean, personal for someone. I figured I was too specific, too hard to relate to and read, possibly, too female. It took a breakup. ["I know there would be a time I could wear them without destroying my feet."] It took a b

### Tokenise

In [8]:
# Defining a function to tokenise a string
def tokenise_string(input_line):
    # Replace URLs with placeholders
    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+\b'
    placeholder = '<URL>'  # Choose a unique placeholder
    urls = re.findall(url_pattern, input_line)
    tokenised_line = re.sub(url_pattern, placeholder, input_line)  # Replace URLs with placeholders

    # Replace curly quotes with straight ones
    tokenised_line = tokenised_line.replace('“', '"').replace('”', '"').replace("‘", "'").replace("’", "'")
    # Separate common punctuation marks with spaces
    tokenised_line = re.sub(r'([.\!?,"\'/()])', r' \1 ', tokenised_line)
    # Add a space before '#'
    tokenised_line = re.sub(r'(?<!\s)#', r' #', tokenised_line)  # Add a space before '#' if it is not already preceded by one
    # Reduce extra spaces by a single space
    tokenised_line = re.sub(r'\s+', ' ', tokenised_line)

    # Replace the placeholders with the respective URLs
    for url in urls:
        tokenised_line = tokenised_line.replace(placeholder, url, 1)

    return tokenised_line

# Tokenising the strings
df_ted_tc['Text'] = df_ted_tc['Text'].apply(tokenise_string)

### Reinspect the row

In [9]:
print(df_ted_tc.at[3319, 'Text'])

The ways of the world often baffle me . I sometimes wonder if I missed the memo about the most basic things . What are you supposed to make for dinner ? What do you talk about in an elevator ? Why do people cut in line ? How do you leave a dinner party without being rude ? Or do you leave it all ? [ " Why are you still here ? " ] My tendency to see the world like I ' m from outer space was a bit of a liability when I was a kid . True story . [ " What ' s a noogie ? " ] But it ' s been helpful in my career . I ' m a cartoonist . When I first started making cartoons for The New Yorker about a decade ago , I kept my ideas light and quirky . And please shout out if you don ' t get any of them , I ' ll explain . [ " The synchronized swim team quits " ] I didn ' t draw anything too personal . [ " Unicorns do exist ! " ] I mean , personal for someone . I figured I was too specific , too hard to relate to and read , possibly , too female . It took a breakup . [ " I know there would be a time I

## Export to a file

In [10]:
df_ted_tc.to_json("corpus/00_sources/cl_st2_ph3_elaine_tc_3.jsonl", orient='records', lines=True)

In [11]:
df_ted_tc.drop(columns=['Text']).to_excel("corpus/00_sources/cl_st2_ph3_elaine_tc_3.xlsx", index=False)

In [16]:
df_ted_tc.drop(columns=['Text']).to_csv("corpus/00_sources/cl_st2_ph3_elaine_tc_3.tsv", sep="\t", index=False)

## Export the target corpus into a directory

### Inspect the `df_ted_tc` DataFrame

In [12]:
df_ted_tc.head()

,Root Directory,File,File Path,Text ID,Year,Word Count NLTK,Text
0,CoTED2_TED_TEDEX_1984_2025,T0004_AM_TED_2019.txt,CoTED2_TED_TEDEX_1984_2025/T0004_AM_TED_2019.txt,t000000,2019,2507,I spent the past three years talking to some o...
1,CoTED2_TED_TEDEX_1984_2025,T0006_AV_TED_2019.txt,CoTED2_TED_TEDEX_1984_2025/T0006_AV_TED_2019.txt,t000001,2019,1115,[This is an improvised talk based on a suggest...
2,CoTED2_TED_TEDEX_1984_2025,T0007_AAB_TED_2019.txt,CoTED2_TED_TEDEX_1984_2025/T0007_AAB_TED_2019.txt,t000002,2019,1734,So one of the most important solutions to the ...
3,CoTED2_TED_TEDEX_1984_2025,T0008_BV_TED_2019.txt,CoTED2_TED_TEDEX_1984_2025/T0008_BV_TED_2019.txt,t000003,2019,1736,"So in the winter of 2012 , I went to visit my ..."
4,CoTED2_TED_TEDEX_1984_2025,T0009_BW_TED_RES_2019.txt,CoTED2_TED_TEDEX_1984_2025/T0009_BW_TED_RES_20...,t000004,2019,1223,For all that ' s ever been said about climate ...


In [13]:
df_ted_tc.shape

(4315, 7)

### Export TED Talks to text files grouped by `Year`

In [15]:
import os

output_dir = 'corpus/01_ted_talks/'

for index, row in df_ted_tc.iterrows():
    text_id = row['Text ID']
    text_content = row['Text']
    year = row['Year']

    year_dir = os.path.join(output_dir, str(year))
    os.makedirs(year_dir, exist_ok=True)

    file_path = os.path.join(year_dir, f"{text_id}.txt")
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(str(text_content))